# Data Recon
Preliminary data fetching and investigation

In [1]:
import pandas as pd
import requests

In [ ]:
# We want to pull data for 3/13 - 3/15. Time is in UTC, so pull 00 UTC for 3/16
date1 = '2026-03-13' # Change these dates for larger timespans
date2 = '2026-03-16'
lsr_url =  "https://mesonet.agron.iastate.edu/cgi-bin/request/gis/lsr.py?wfo=ALL&sts=" + date1 + "T00:00Z&ets=" + date2 + "T00:00Z&fmt=csv"
print(lsr_url)

https://mesonet.agron.iastate.edu/cgi-bin/request/gis/lsr.py?wfo=ALL&sts=2026-03-13T00:00Z&ets=2026-03-16T00:00Z&fmt=csv


In [25]:
url = r"https://mesonet.agron.iastate.edu/cgi-bin/request/gis/lsr.py?wfo=ALL&sts=2026-03-13T00:00Z&ets=2026-03-16T00:00Z&fmt=csv"
lsr = pd.read_csv(url, engine='python', on_bad_lines='skip') 
lsr_df = pd.DataFrame(lsr)
lsr_df.head()

,VALID,VALID2,LAT,LON,MAG,WFO,TYPECODE,TYPETEXT,CITY,COUNTY,STATE,SOURCE,REMARK,UGC,UGCNAME,QUALIFIER
0,202603130000,2026/03/13 00:00,39.45,-74.57,0.0,PHI,S,SNOW,Atlantic City Internati,Atlantic,NJ,ASOS,00z Snow Observation. Trace.,NJC001,Atlantic,M
1,202603130000,2026/03/13 00:00,40.65,-75.44,0.0,PHI,S,SNOW,Lehigh Valley Internati,Lehigh,PA,ASOS,00z Snow Observation. Trace.,PAC077,Lehigh,M
2,202603130000,2026/03/13 00:00,39.68,-75.61,0.0,PHI,S,SNOW,New Castle County Airpo,New Castle,DE,ASOS,00z Snow Observation.,DEC003,New Castle,M
3,202603130000,2026/03/13 00:00,39.88,-75.22,0.0,PHI,S,SNOW,Philadelphia Internatio,Philadelphia,PA,ASOS,00z Snow Observation. Trace.,PAC101,Philadelphia,M
4,202603130000,2026/03/13 00:00,40.28,-74.81,0.0,PHI,S,SNOW,Trenton Mercer Airport,Mercer,NJ,ASOS,00z Snow Observation. Trace.,NJC021,Mercer,M


## LSR FETCHING AND INSPECTION

Upon initial reading of lsr_df, it came back with ParserWarnings. Upon investigating CSV, there were LSRs that were in marine zone areas (Great Lakes) that are not needed, and so it was decided to skip rows 617 and 794. Finding the acual rows was found with the 'on_bad_lines'='warn' parameter, and rows were investigated with a 'good' row. Since it was 2 rows, decided to skip them rather than to manually repair them.

#### Problem
Finding what lines in the LSR CSV is giving us problems
Appears to be lines 617 and 794. Investigating

In [9]:
response = requests.get(url)
lines = response.text.splitlines()
print(lines[2])

print(lines[617])
print(lines[794])

202603130000,2026/03/13 00:00,40.65,-75.44,0.0,PHI,S,SNOW,Lehigh Valley Internati,Lehigh,PA,ASOS,00z Snow Observation. Trace.,PAC077,Lehigh,M
202603131040,2026/03/13 10:40,45.62,-86.66,44.0,MQT,N,NON-TSTM WND GST,Fairport, MI,LMZ250,MI,Mesonet,GLOS Weather Station FPTM4_ 6 S Fayette.,LMZ250,5NM East of a line from Fairport MI to Rock Island Passage,M
202603131150,2026/03/13 11:50,46.68,-85.97,40.0,MQT,N,NON-TSTM WND GST,Grand Marais, MI,LSZ251,MI,Mesonet,GLOS Weather Station GRMM4_ 1 NNE Grand Marais.,LSZ251,Grand Marais to Whitefish Point MI,M


In [74]:
# Column names and types
print(lsr_df.dtypes, '\n')

# Finding NULL values
print(lsr_df.isnull().sum(), '\n') 

# Getting unique TYPECODES
print('Unique TYPECODES in this dataset are:')
print(lsr_df['TYPECODE'].unique(), '\n')

# Filtering wind-only events and then summary of MAG column
wind_codes = ['D','O','A','G','N'] # Any severity 'wind damage' codes
lsr_wind = lsr_df[lsr_df['TYPECODE'].isin(wind_codes)]
print(lsr_wind['MAG'].describe(), '\n')
print(lsr_wind['MAG'].isnull().sum(), '\n')

# Finding null counts per wind TYPECODE
print(lsr_wind.groupby('TYPECODE').apply(lambda x: x.isnull().sum()))


VALID          int64
VALID2        object
LAT          float64
LON          float64
MAG          float64
WFO           object
TYPECODE      object
TYPETEXT      object
CITY          object
COUNTY        object
STATE         object
SOURCE        object
REMARK        object
UGC           object
UGCNAME       object
QUALIFIER     object
dtype: object 

VALID          0
VALID2         0
LAT            0
LON            0
MAG          691
WFO            0
TYPECODE       0
TYPETEXT       0
CITY           0
COUNTY         0
STATE          0
SOURCE         0
REMARK       908
UGC          133
UGCNAME      133
QUALIFIER    691
dtype: int64 

Unique TYPECODES in this dataset are:
['S' 'N' 'x' 'O' 'Z' 'W' 'R' 'E' 'F' '2' 'D' 'G' 'V' 'J' 'P' 'q' 'U' '5'
 'H' 'A' 'M' 'T' 'C'] 

count    3091.000000
mean       59.044646
std         9.456147
min        30.000000
25%        53.000000
50%        59.000000
75%        64.000000
max       141.000000
Name: MAG, dtype: float64 

557 

          VALID  VALID2 